In [1]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [5]:
import torch
from PIL import Image

image_paths = finna.id.apply(lambda x: f"/content/drive/MyDrive/MoMA_vs_Finna/finna_imgs/{x}.jpg")
# List of material classes is set empirically by looking through the image collection and considering the mediums present in the MoMA dataset
classes = ['wood', 'metal', 'glass', 'ceramics', 'textile', 'plastic', 'paper']
batch_size = 32
probs = []

for i in range(0, len(image_paths), batch_size):
  i_end = i + batch_size if i + batch_size < len(image_paths) else len(image_paths)
  batch_image_paths = image_paths[i:i_end]

  batch_images = [Image.open(path) for path in batch_image_paths]

  inputs = processor(text=classes, images=batch_images, return_tensors="pt", padding=True)
  outputs = model(**inputs)
  batch_logits = outputs.logits_per_image
  batch_probs = batch_logits.softmax(dim=1)
  probs.extend(batch_probs.tolist())

In [11]:
def predict_class(probs_per_image):
  max_p = np.max(probs_per_image)
  if max_p > 0.5:
    max_id = np.argmax(probs_per_image)
    return classes[max_id].capitalize()
  else:
    return "Other"

predictions = [predict_class(row) for row in probs]

finna = pd.read_csv("/content/drive/MyDrive/MoMA_vs_Finna/finna_designs.csv", index_col=0)
finna['Medium'] = predictions
finna['Medium'].value_counts()

,count
Medium,
Ceramics,406
Glass,381
Other,275
Paper,215
Wood,181
Metal,96
Textile,91
Plastic,17


In [ ]:
finna.to_csv("finna_upd_medium.csv")

In [ ]:
finna[finna.Medium.isin(['Ceramics', 'Glass', 'Paper', 'Wood', 'Metal', 'Textile'])].to_csv("finna_six_mediums.csv")